<a href="https://colab.research.google.com/github/sotesh1516/deep-ML/blob/main/CNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import PIL
import numpy
import IPython
from io import BytesIO
from numpy.lib.stride_tricks import sliding_window_view
images = []
targets = []
for i in range(1,5):
  im = PIL.Image.open(f'/content/drive/MyDrive/Colab Notebooks/{i}.jpg').convert('L')
  bio = BytesIO() #treat a chuck of mem as file, since PIL only writes to file obj
  im.save(bio, format="png")
  display(IPython.display.Image(data=bio.getvalue(), format='png'))
  images.append(numpy.asarray(im)/255) #squash from 0-1
  targets.append(i/4) #squash 0.25 to 1

In [4]:
numpy.random.seed(5814)

In [12]:
#a 5x5 conv, no bias
w1 = numpy.random.randn(5,5)
#a 3x3 conv, no bias. will follow a max pool layer
w2 = numpy.random.randn(3,3)
print(w2)
#finally fcl, will follow another max pool layer
w3 = numpy.random.randn(4,4) # row of fc is pre-determined since input dim, conv_kernel size, max_pool size are hard-coded

[[-1.20577926  1.04722631 -0.41620052]
 [-1.06186779 -0.83242244 -0.64899835]
 [-0.83145136  1.6665197   1.63621231]]


In [6]:
num_epoch = 1000
learning_rate = 0

loss_before_training = 0
loss_after_training = 0

final_output, start_output = numpy.array([[]]), numpy.array([[]])

In [7]:
w4 = numpy.random.randn(2,2)

In [8]:
# this won't work if kernel > input
def my_2dconv(X, weights):
  rows, cols = X.shape
  kernel_rows, kernel_cols = weights.shape

  output_rows = rows - kernel_rows + 1
  output_cols = cols - kernel_cols + 1
  output = numpy.zeros((output_rows, output_cols))

  for i in range(output_rows):
    for j in range(output_cols):
      window = X[i:i+kernel_rows,j:j+kernel_cols]
      product = (window * weights).sum()
      output[i, j] = product


  return output

def my_2dconv_backward_input(X, weights, upstream_grad): #return dL/dX
  rows, cols = X.shape
  kernel_rows, kernel_cols = weights.shape

  rotated_kernel = numpy.fliplr(numpy.flipud(weights))

  row_padding = kernel_rows - 1
  col_padding = kernel_cols - 1

  padded_upstream = numpy.pad(upstream_grad, ((row_padding, row_padding),
   (col_padding, col_padding)), mode='constant')


  return my_2dconv(padded_upstream, rotated_kernel)


def my_2dconv_backward_weights(X, weights, upstream_grad): #return dL/dW
  rows, cols = X.shape
  kernel_rows, kernel_cols = weights.shape
  output = numpy.zeros((kernel_rows, kernel_cols))

  for i in range(kernel_rows):
    for j in range(kernel_cols):

      input_h_start = i #assuming stride = 1
      input_h_end = input_h_start + kernel_rows
      input_w_start = j
      input_w_end = input_w_start + kernel_cols


      current_weight_window = X[input_h_start:input_h_end, input_w_start:input_w_end]
      output[i,j] += current_weight_window * upstream_grad


  return output


def max_pool_2x2(X):
  rows, cols = X.shape

  output_rows = ((rows - 2)/2)+1
  output_cols = ((cols - 2)/2)+1
  output = numpy.zeros((output_rows, output_cols))

  for i in range(output_rows):
    for j in range(output_cols):
      window = X[i:i+2,j:j+2]
      output[i,j] = numpy.max(window)


  return output

def max_pool_2x2_backward(X, upstream_grad):
  rows, cols = X.shape
  upstream_rows, upstream_cols = upstream_grad.shape
  stride = 2
  pool_size = 2

  output = numpy.zeros((rows, cols))

  for i in range(upstream_rows):
    for j in range(upstream_cols):

      pool_h_start = i * stride
      pool_h_end = pool_h_start + pool_size
      pool_w_start = j * stride
      pool_w_end = pool_w_start + pool_size

      current_input_window = X[pool_h_start:pool_h_end, pool_w_start:pool_w_end]
      mask = (current_input_window == numpy.max(current_input_window))

      output[pool_h_start:pool_h_end, pool_w_start:pool_w_end] += mask * upstream_grad[i,j]


  return output

def relu(X):
  return numpy.max(X, 0)

def relu_gradient(X): # no need to deal with any other input since all relu needs is the upstream_grad which in this case is X
  return (X > 0).astype(float)

def sigmoid(X):
  return 1 / (1 + numpy.exp(-X))

def sigmoid_gradient(X):
  return sigmoid(X) * (1 - sigmoid(X))

In [11]:
for i in range(len(images)):

  conv_1 = my_2dconv(i, w1)
  relu = relu(conv_1)
  max_pool_1 = max_pool_2x2(relu)
  conv_2 = my_2dconv(max_pool_1, w2)
  max_pool_2 = max_pool_2x2(conv_2)
  fc = max_pool_2 @ w3 # matrix mul, (1 x 4) @ (4 x 4)
  output_from_network = sigmoid(fc)

  loss = None
  loss_before_training += loss
  start_output = numpy.append(start_output, output_from_network)

(16, 16)
(16, 16)
(16, 16)
(16, 16)
